In [1]:
import re
import time
from colorama import Fore
import openpyxl
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.chrome.options import Options
import pandas as pd
from selenium import webdriver
from selenium.common.exceptions import WebDriverException
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import StaleElementReferenceException
import random
from selenium.common.exceptions import ElementClickInterceptedException
from selenium.common.exceptions import NoSuchElementException
import math

In [2]:
link='https://www.arzt-auskunft.de/augenheilkunde/'

In [77]:
options = Options()

options.add_argument('--disable-application-cache')
options.add_argument('--disable-blink-features=AutomationControlled')
options.add_argument("--disable-cookies")
options.add_argument("--disable-extensions")
options.add_argument('--no-sandbox')
options.add_argument("--disable-extensions")
options.add_argument("start-maximized")
options.add_argument('--disable-gpu')
# options.add_argument("--headless")  
# options.add_argument("--disable-gpu")
options.add_argument("--disable-dev-shm-usage")
options.add_argument('--ignore-urlfetcher-cert-requests')
options.add_argument('--no-first-run')
options.add_argument("--disable-popup-blocking") 


driver = webdriver.Chrome(options=options)

In [5]:
driver.get(link)

In [36]:
urlstoscrap=[]
card_title=[]


In [37]:
page=BeautifulSoup(driver.page_source,'html.parser')
cards=page.find_all('div',class_='card dl mb-3')

In [38]:
for firstpagecard in cards:
    urlstoscrap.append(firstpagecard.find('a')['href'])
    card_title.append(firstpagecard.find('h3', class_='card-title').text.replace('\n', '').replace('\xa0', '').strip())

In [39]:
for e in range(2,120):
    driver.get('https://www.arzt-auskunft.de/augenheilkunde/'+str(e)+'/')
    time.sleep(2)
    page2=BeautifulSoup(driver.page_source,'html.parser')
    cards2=page2.find_all('div',class_='card dl mb-3')
    for card in cards2:
       urlstoscrap.append(card.find('a')['href'])
       card_title.append(card.find('h3', class_='card-title').text.replace('\n', '').replace('\xa0', '').strip())
    



In [65]:
firstdata=({
    'title':card_title,
    'urls':urlstoscrap
})

In [66]:
firstdata=pd.DataFrame(firstdata).drop_duplicates()

In [67]:
firstdata['name']=None
firstdata['medicalSpecialty']=None
firstdata['title_h3']=None
firstdata['adress']=None
firstdata['tel_number']=None
firstdata['fax_number']=None
firstdata['page_url']=None


In [81]:
for index, nombre in enumerate(firstdata['name']):
    if nombre is None:
      link=firstdata['urls'][index]
      driver.get(link)
      time.sleep(2)
      soup=BeautifulSoup(driver.page_source,'html.parser')
      try:
         prename=soup.find('h1', attrs={'itemprop': 'name'}).text.replace('\n','').strip()
         if prename:
            firstdata['name'][index]=prename
      except AttributeError:
        pass 
      try:
         premedicalSpeciality=soup.find('span', attrs={'itemprop': 'medicalSpecialty'}).text.replace('\n','').strip()
         if premedicalSpeciality:
            firstdata['medicalSpecialty'][index]=premedicalSpeciality
      except AttributeError:
        pass 
      try:
         pretitle_h3=soup.find('h3').text.replace('\n','').strip()
         if pretitle_h3:
            firstdata['title_h3'][index]=pretitle_h3
      except AttributeError:
        pass 
      try:
         preadress=soup.find('div', attrs={'itemprop': 'address'}).text.replace('\n',' ').strip()
         if preadress:
            firstdata['adress'][index]=preadress
      except AttributeError:
        pass 
      try:
         pretel_number=soup.find('span', attrs={'itemprop': 'telephone'}).text.replace('\n','').strip()
         if pretel_number:
            firstdata['tel_number'][index]=pretel_number
      except AttributeError:
        pass 
      try:
         prefax_number=soup.find('span', attrs={'itemprop': 'fax'}).text.replace('\n','').strip()
         if prefax_number:
            firstdata['fax_number'][index]=prefax_number
      except AttributeError:
        pass 
      try:
         prepage_url=soup.find('span', attrs={'itemprop': 'url'}).text.replace('\n','').strip()
         if prepage_url:
            firstdata['page_url'][index]=prepage_url
      except AttributeError:
        pass 
      


C:\Users\juana\AppData\Local\Temp\ipykernel_22164\3898522789.py:10: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  firstdata['name'][index]=prename
C:\Users\juana\AppData\Local\Temp\ipykernel_22164\3898522789.py:16: FutureWarning: ChainedAssi

In [80]:
index

10589

In [83]:
firstdata.to_csv('arzt-auskunft.csv',index=False)
firstdata.to_excel('arzt-auskunft.xlsx',index=False)